# Adaptive Compute SSM — Phase 3: Expert Birth/Death Lifecycle
## Self-Organising Expert Pool · Marginal Utility Index · Tournament Selection · Lifecycle Management

**Phase 3 changes vs Phase 2:**
- `MoERouter` — dynamic centroid add/remove (`add_centroid`, `remove_centroid`)
- `MoEProcessingLayer` — dynamic expert pool via `register_module` + `_expert_ids`
- `TournamentExpert` — sequential multi-type tournament during trial (grace) period
- `ExpertLifecycleManager` — rolling per-expert statistics, birth/death decisions
- Training loop — lifecycle hooks, optimizer rebuild on pool change

### Phase roadmap
| Phase | What we add |
|-------|-------------|
| 1 | SSM + monolithic MLP + depth gate + KS anchoring + log-norm regularization |
| 2 | Retrieval-based MoE router + heterogeneous expert types |
| **3 (this notebook)** | **Expert birth/death lifecycle (marginal utility index)** |
| 4 | Inference-time Smirnov depth control (μ shift) |
| 5 | Scale to full pre-training corpus |


In [ ]:
# Install all dependencies first
# comet_ml MUST be installed and imported BEFORE PyTorch
!pip install -q comet_ml datasets transformers tokenizers tqdm
# Install FAISS — GPU variant when CUDA is available, CPU otherwise
import subprocess, sys
try:
    _has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except FileNotFoundError:
    _has_gpu = False
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "faiss-gpu" if _has_gpu else "faiss-cpu"])


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  IMPORTANT: comet_ml MUST be imported BEFORE torch / transformers       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import comet_ml
from comet_ml import Experiment
from comet_ml.integration.pytorch import log_model

from google.colab import userdata
COMET_KEY = userdata.get('COMET_KEY')

experiment = Experiment(
    api_key      = COMET_KEY,
    project_name = 'adaptive-ssm',
    workspace    = 'irsotarriva'
)

import math, random
from collections import defaultdict, deque
import numpy as np
import faiss
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from transformers import AutoTokenizer
from datasets import load_dataset
from huggingface_hub import login

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
from google.colab import userdata
HF_KEY = userdata.get('HF_KEY')
login(token=HF_KEY)


---
## Architecture Overview — Phase 3 Expert Lifecycle

```
token_id ─► Embedding ─► e_t ──────────────────────────────────────────────────────────────┐
                                                                                            │
                    h_{t-1} ──► InputLayer ──► z_0                                         │
                                                │                                          │
                              ┌─────────────────┤                                          │
                              │  for k in range(max_depth):                               │
                              │    s_k = GateMLP(z_k)   → [KS anchoring loss]            │
                              │    z_{k+1} = DynMoELayer(z_k)   ← dynamic expert pool    │
                              └─────────────────┤                                          │
                                                │                                          │
                              OutputLayer ──► logits + h_t ──────────────────────────────►│
                                                                                            │
ExpertLifecycleManager ─────────────────────────────────────────────────────────────────────┘
  Tracks: query_counts[eid], loss_credits[eid] (rolling window)
  Birth:  U(e) = mean_loss / log(1 + query_count) > birth_threshold → spawn TournamentExpert
  Death:  query_count < death_threshold → prune
  Tournament: sequential gelu→silu→poly, promote winner after grace_period steps
```

**Key invariant** (unchanged): SSM state is read once (InputLayer) and written once (OutputLayer).
The dynamic expert pool does NOT change this contract.


---
## 1. Heterogeneous Expert Types  *(unchanged from Phase 2)*

Same interface as Phase 2: `forward(z) → Δ` where Δ is a delta (residual).
All experts zero-init their output projection so they start as identity.


In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalisation (Zhang & Sennrich, 2019)."""
    def __init__(self, d: int, eps: float = 1e-8):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d))
        self.eps   = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.scale * x / rms


class MLPExpert(nn.Module):
    """
    SwiGLU expert: F.silu(W_gate(z_n)) * W_up(z_n) projected by W_down.
    Predicts a DELTA (residual) — does NOT include a skip connection.
    The skip connection is applied at the MoEProcessingLayer level.
    """
    def __init__(self, d_internal: int, d_hidden: int):
        super().__init__()
        self.norm   = RMSNorm(d_internal)
        self.W_gate = nn.Linear(d_internal, d_hidden, bias=False)
        self.W_up   = nn.Linear(d_internal, d_hidden, bias=False)
        self.W_down = nn.Linear(d_hidden,   d_internal, bias=False)
        nn.init.zeros_(self.W_down.weight)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        z_n = self.norm(z)
        return self.W_down(F.silu(self.W_gate(z_n)) * self.W_up(z_n))


class PolynomialExpert(nn.Module):
    """
    Quadratic feature expansion followed by a linear projection.
    Computes [z_norm, z_norm\u00b2] \u2192 Linear \u2192 delta.
    """
    def __init__(self, d_internal: int):
        super().__init__()
        self.norm = RMSNorm(d_internal)
        self.proj = nn.Linear(d_internal * 2, d_internal, bias=False)
        nn.init.zeros_(self.proj.weight)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        z_n = self.norm(z)
        return self.proj(torch.cat([z_n, z_n.pow(2)], dim=-1))


---
## 2. Retrieval-Based MoE Router  *(modified: dynamic centroid management)*

Phase 3 changes:
- `add_centroid(vec)` — appends a new centroid row, re-registers `nn.Parameter`
- `remove_centroid(idx)` — removes centroid at position idx, re-registers
- `forward` dynamically clips `top_k` to `min(top_k, n_experts)` so the pool can start small


In [ ]:
class MoERouter(nn.Module):
    """
    KNN-based differentiable router with fixed-size preallocated centroid pool.

    Phase 3 additions:
      add_centroid(vec)    \u2014 activate next inactive slot
      remove_centroid(idx) \u2014 deactivate slot at position idx

    Phase 3 (DDP) changes:
      Centroids are a fixed (max_experts, d_internal) Parameter;
      expert_active is a registered boolean buffer tracking which slots are live.
      This keeps the parameter set constant across the training run so that DDP
      gradient buckets are never rebuilt.

    Phase 3 (FAISS) changes:
      When n_active > faiss_threshold, forward() uses a FAISS IndexFlatL2 to
      retrieve top_k*4 candidates (detached), then computes exact differentiable
      distances only over those candidates so gradients still flow.
      Call rebuild_faiss_index() periodically from the training loop.

    Phase 3 (null route) changes:
      null_logit is a learned scalar that competes with expert RBF scores in a
      joint softmax. When null is in the top-k, it contributes zero delta but
      its weight dilutes expert contributions, enabling smooth bypass.
      forward() returns (weights, topk_idx, distances, all_weights_full,
      null_weight_mean) where topk_idx value == n_active means null route.

    forward() always clips top_k to min(self.top_k, n_active + 1) to allow
    null to compete even when the pool is small.
    """

    def __init__(
        self,
        d_internal    : int,
        n_experts     : int,
        top_k         : int,
        temperature   : float = 1.0,
        max_experts   : int   = 64,
        faiss_threshold: int  = 32,
    ):
        super().__init__()
        self.d_internal      = d_internal
        self.top_k           = top_k
        self.temperature     = temperature
        self.max_experts     = max_experts
        self.faiss_threshold = faiss_threshold

        self.query_proj = nn.Linear(d_internal, d_internal, bias=False)

        # Fixed-size centroid tensor \u2014 always (max_experts, d_internal)
        c = torch.zeros(max_experts, d_internal)
        if n_experts > 0:
            c[:n_experts] = F.normalize(torch.randn(n_experts, d_internal), dim=-1)
        self.centroids = nn.Parameter(c)

        # Learned null route \u2014 competes with expert RBF scores in joint softmax
        self.null_logit = nn.Parameter(torch.zeros(1))

        # Active-slot mask \u2014 registered buffer so it is saved in state_dict
        active = torch.zeros(max_experts, dtype=torch.bool)
        active[:n_experts] = True
        self.register_buffer('expert_active', active)

        # FAISS index \u2014 rebuilt periodically, not saved in state_dict
        self._faiss_index            = None
        self._faiss_active_snapshot  = []   # active slot list at last rebuild

    @property
    def n_experts(self) -> int:
        return int(self.expert_active.sum().item())

    @property
    def active_slots(self) -> list:
        """Sorted list of currently active global slot indices."""
        return self.expert_active.nonzero(as_tuple=True)[0].tolist()

    def _next_inactive_slot(self):
        inactive = (~self.expert_active).nonzero(as_tuple=True)[0]
        return inactive[0].item() if len(inactive) > 0 else None

    # \u2500\u2500 Dynamic pool management \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    def add_centroid(self, new_vec: torch.Tensor, slot: int = None) -> int:
        """Activate next inactive slot (or specified slot) with new_vec.
        Returns the slot index that was activated."""
        with torch.no_grad():
            if slot is None:
                slot = self._next_inactive_slot()
            if slot is None:
                raise RuntimeError("Expert pool is full (max_experts reached)")
            v = F.normalize(new_vec.to(self.centroids.device), dim=0)
            self.centroids.data[slot] = v
            self.expert_active[slot]  = True
        return slot

    def remove_centroid(self, slot: int):
        """Deactivate slot (mark as inactive)."""
        with torch.no_grad():
            self.expert_active[slot] = False

    # \u2500\u2500 FAISS index management \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    def rebuild_faiss_index(self):
        """Rebuild FAISS IndexFlatL2 from current active centroids.
        Only builds an index when n_active > faiss_threshold.
        Call this from the training loop every faiss_rebuild_interval steps."""
        active = self.active_slots
        if len(active) <= self.faiss_threshold:
            self._faiss_index = None
            return
        vecs = self.centroids.data[active].cpu().float().contiguous().numpy()
        idx  = faiss.IndexFlatL2(self.d_internal)
        idx.add(vecs)
        self._faiss_index           = idx
        self._faiss_active_snapshot = list(active)

    # \u2500\u2500 Forward \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    def forward(self, z: torch.Tensor):
        """
        z: (batch, d_internal)
        Returns:
          weights         (B, actual_k)  \u2014 softmax weights incl. null
          topk_idx        (B, actual_k)  \u2014 positions in active set; n_active = null
          distances       (B, n_active)  \u2014 L2 distances for repulsion loss
          all_weights_full(B, n_active+1)\u2014 full routing distribution (w/ gradient)
          null_weight_mean float         \u2014 mean null weight this step

        actual_k = min(self.top_k, n_active + 1), clipped so null competes.
        Returns empty tensors if pool is empty.
        """
        active = self.active_slots
        n = len(active)
        if n == 0:
            dev = z.device; B = z.shape[0]
            return (torch.zeros(B, 0, device=dev),
                    torch.zeros(B, 0, dtype=torch.long, device=dev),
                    torch.zeros(B, 0, device=dev),
                    torch.zeros(B, 1, device=dev),
                    0.0)

        B            = z.shape[0]
        actual_k     = min(self.top_k, n + 1)   # +1 for null route
        q            = self.query_proj(z)
        dev          = z.device
        active_t     = torch.tensor(active, device=dev, dtype=torch.long)
        active_c     = self.centroids[active_t]          # (n, d) \u2014 with grad

        if n > self.faiss_threshold and self._faiss_index is not None:
            # \u2500\u2500 Approximate path: FAISS retrieves candidates, exact for top-k \u2500\u2500
            n_cand      = min(actual_k * 4, n)
            q_np        = q.detach().cpu().float().contiguous().numpy()
            _, fi_local = self._faiss_index.search(q_np, n_cand)
            snap        = self._faiss_active_snapshot
            active_set  = set(active)

            # Map FAISS local \u2192 global slot, filter stale / invalid entries
            cand_rows = []
            for b in range(B):
                row = [snap[i] for i in fi_local[b]
                       if 0 <= i < len(snap) and snap[i] in active_set]
                if not row:
                    row = active[:n_cand]
                cand_rows.append(row)

            max_c  = max(len(r) for r in cand_rows)
            padded = [r + [active[0]] * (max_c - len(r)) for r in cand_rows]
            cand_g = torch.tensor(padded, device=dev, dtype=torch.long)  # (B, max_c)

            # Exact distances over candidates \u2014 gradients flow through centroids
            cand_c = self.centroids[cand_g]                    # (B, max_c, d)
            cand_d = (q.unsqueeze(1) - cand_c).norm(dim=-1)   # (B, max_c)

            # Mask padding slots
            pad_mask = torch.zeros(B, max_c, dtype=torch.bool, device=dev)
            for b, row in enumerate(cand_rows):
                if len(row) < max_c:
                    pad_mask[b, len(row):] = True

            # RBF for all candidates; zero out padded positions
            rbf_cand = torch.exp(-cand_d.pow(2) / (self.temperature ** 2 + 1e-8))
            rbf_cand = rbf_cand.masked_fill(pad_mask, 0.0)

            # Softmax over [candidates | null_logit]
            null_s  = self.null_logit.expand(B, 1)
            scores  = torch.cat([rbf_cand, null_s], dim=-1)    # (B, max_c+1)
            all_w_c = F.softmax(scores, dim=-1)                 # (B, max_c+1)

            cand_w  = all_w_c[:, :-1]   # (B, max_c)
            null_w  = all_w_c[:, -1]    # (B,)

            # Top-k by weight (null at column max_c)
            topk_w, topk_ci2 = all_w_c.topk(actual_k, dim=-1, largest=True)

            # Build slot_to_pos mapping
            slot_to_pos = torch.full((self.max_experts,), -1, device=dev, dtype=torch.long)
            for i, s in enumerate(active):
                slot_to_pos[s] = i

            is_null     = (topk_ci2 >= max_c)                  # (B, actual_k)
            safe_ci     = topk_ci2.clamp(max=max_c - 1)
            topk_global = cand_g.gather(1, safe_ci)            # (B, actual_k)
            topk_pos    = slot_to_pos[topk_global]
            topk_idx    = topk_pos.masked_fill(is_null, n)

            weights     = topk_w

            # Reconstruct full all_weights (B, n+1) for entropy computation
            all_w_full = torch.zeros(B, n + 1, device=dev)
            pos_of_cand = slot_to_pos[cand_g]                  # (B, max_c)
            valid        = (pos_of_cand >= 0) & ~pad_mask
            pos_clamped  = pos_of_cand.clamp(min=0)
            all_w_full.scatter_add_(1, pos_clamped, cand_w * valid.float())
            all_w_full[:, n] = null_w

            null_weight_mean = null_w.mean().item()

            # Full distances over active set (used by repulsion loss)
            distances = (q.unsqueeze(1) - active_c.unsqueeze(0)).norm(dim=-1)
            return weights, topk_idx, distances, all_w_full, null_weight_mean

        # \u2500\u2500 Exact path \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
        diff      = q.unsqueeze(1) - active_c.unsqueeze(0)
        distances = diff.norm(dim=-1)                           # (B, n)

        # RBF scores for all active experts
        rbf_all   = torch.exp(-distances.pow(2) / (self.temperature ** 2 + 1e-8))

        # Compete with null route via joint softmax
        null_s      = self.null_logit.expand(B, 1)             # (B, 1)
        scores      = torch.cat([rbf_all, null_s], dim=-1)     # (B, n+1)
        all_weights = F.softmax(scores, dim=-1)                 # (B, n+1)

        topk_weights, topk_idx = all_weights.topk(actual_k, dim=-1, largest=True)
        # topk_idx values 0..n-1 are expert positions; n is the null route

        null_weight_mean = all_weights[:, n].mean().item()
        return topk_weights, topk_idx, distances, all_weights, null_weight_mean


---
## 3. MoE Processing Layer  *(modified: dynamic expert management)*

Phase 3 changes:
- Experts tracked by integer ID (`_expert_ids`) rather than list position
- `register_module(f'expert_{eid}', module)` for clean add/remove with PyTorch bookkeeping
- `add_expert(expert, centroid_vec)` → new eid; `remove_expert(eid)` → shrinks pool
- `replace_expert(eid, new_module)` → used by tournament promotion (same centroid, new weights)
- `forward` iterates `_expert_ids` and returns `expert_ids` list in routing_info for lifecycle tracking


In [ ]:
class MoEProcessingLayer(nn.Module):
    """
    Retrieval-based MoE processing layer with fixed-size preallocated expert pool.

    Phase 3 (DDP) changes:
      All max_experts expert modules are registered at construction time as
      expert_0 \u2026 expert_{max_experts-1}.  Inactive experts receive no gradient
      because they are never in the routing path.  Birth / death only flip the
      router's expert_active mask and swap module weights; the parameter set is
      constant, so DDP gradient buckets are never rebuilt.

    Phase 3 (dispatch) changes:
      MLP experts are dispatched via grouped batched einsum (one pass for all
      SwiGLU MLPs). _type_groups: dict[str, list[int]] maps type \u2192 active slot IDs.
      Updated whenever the pool changes.
      PolynomialExpert and TournamentExpert instances remain sequential.

    Phase 3 (null route) changes:
      Router returns a 5-tuple; null route (topk_idx == n_active) contributes
      a zero delta via an appended zero row in all_deltas.
      routing_info includes all_weights_grad (with gradient) for entropy loss
      and null_weight_mean for Comet logging.

    routing_info includes expert_ids (active slot IDs) and z_input for the
    lifecycle manager's input buffer.
    """

    _TYPE_CYCLE = ['gelu', 'silu', 'poly']

    def __init__(
        self,
        d_internal    : int,
        d_expert      : int,
        n_experts_init: int,
        top_k         : int,
        temperature   : float = 1.0,
        max_experts   : int   = 64,
        faiss_threshold: int  = 32,
    ):
        super().__init__()
        self.d_internal  = d_internal
        self.d_expert    = d_expert
        self.top_k       = top_k
        self.max_experts = max_experts

        self.router = MoERouter(
            d_internal, 0, top_k, temperature,
            max_experts=max_experts, faiss_threshold=faiss_threshold
        )

        # Preallocate ALL max_experts slots at construction time
        for i in range(max_experts):
            t = self._TYPE_CYCLE[i % len(self._TYPE_CYCLE)]
            self.register_module(f'expert_{i}', self._make_expert(t))

        # expert_labels indexed by slot ID (size max_experts)
        self.expert_labels: list = [''] * max_experts

        # Activate initial n_experts_init slots
        for i in range(n_experts_init):
            t = self._TYPE_CYCLE[i % len(self._TYPE_CYCLE)]
            c = F.normalize(torch.randn(d_internal), dim=0)
            self.router.add_centroid(c, slot=i)
            self.expert_labels[i] = t

        # Type-grouped dispatch dict \u2014 rebuilt on pool changes
        self._type_groups: dict = {}
        self._rebuild_type_groups()

    # \u2500\u2500 Factory \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    def _make_expert(self, expert_type: str) -> nn.Module:
        if expert_type == 'poly':
            return PolynomialExpert(self.d_internal)
        return MLPExpert(self.d_internal, self.d_expert)

    # \u2500\u2500 Type-group management \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    def _rebuild_type_groups(self):
        """Rebuild _type_groups from current active slots.
        Called after every pool mutation (add / remove / replace)."""
        groups: dict = {'mlp': [], 'poly': [], 'tournament': []}
        for slot in self.router.active_slots:
            exp = self.get_expert(slot)
            if isinstance(exp, TournamentExpert):
                groups['tournament'].append(slot)
            elif isinstance(exp, PolynomialExpert):
                groups['poly'].append(slot)
            elif isinstance(exp, MLPExpert):
                groups['mlp'].append(slot)
        self._type_groups = {k: v for k, v in groups.items() if v}

    # \u2500\u2500 Pool properties \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    @property
    def n_experts(self) -> int:
        return self.router.n_experts

    @property
    def _expert_ids(self) -> list:
        """Active slot IDs \u2014 backward-compatible name used by training loop."""
        return self.router.active_slots

    def get_expert(self, slot: int) -> nn.Module:
        return self.get_submodule(f'expert_{slot}')

    def get_all_experts(self) -> list:
        return [self.get_expert(s) for s in self.router.active_slots]

    # \u2500\u2500 Dynamic management \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    def add_expert(
        self,
        expert      : nn.Module,
        centroid_vec: torch.Tensor,
        label       : str = None,
    ) -> int:
        """Activate next inactive slot with given expert and centroid.
        Returns the slot ID that was activated."""
        slot = self.router.add_centroid(centroid_vec)   # activates slot, returns ID
        self.register_module(f'expert_{slot}', expert)
        self.expert_labels[slot] = label or f'expert_{slot}'
        self._rebuild_type_groups()
        return slot

    def remove_expert(self, slot: int):
        """Deactivate slot and reinitialise its module to a default expert."""
        self.router.remove_centroid(slot)
        self.expert_labels[slot] = ''
        # Re-initialise with a default expert so the parameter is well-defined
        t = self._TYPE_CYCLE[slot % len(self._TYPE_CYCLE)]
        self.register_module(f'expert_{slot}', self._make_expert(t))
        self._rebuild_type_groups()

    def replace_expert(self, slot: int, new_expert: nn.Module,
                       new_label: str = None):
        """Replace module at slot in-place (same centroid, new weights)."""
        self.register_module(f'expert_{slot}', new_expert)
        if new_label is not None:
            self.expert_labels[slot] = new_label
        self._rebuild_type_groups()

    # \u2500\u2500 Batched dispatch \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    def _batched_dispatch(self, z: torch.Tensor, active_slots: list) -> torch.Tensor:
        """
        Compute expert outputs using grouped batched einsum for SwiGLU MLP types.
        PolynomialExpert and TournamentExpert instances run sequentially.
        Returns all_deltas: (batch, n_active, d_internal).
        """
        B, D  = z.shape
        n     = len(active_slots)
        all_d = torch.zeros(B, n, D, device=z.device, dtype=z.dtype)
        pos   = {s: i for i, s in enumerate(active_slots)}

        # \u2500\u2500 Batched SwiGLU MLPs \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
        m_slots = self._type_groups.get('mlp', [])
        if m_slots:
            exps     = [self.get_expert(s) for s in m_slots]
            # W_gate / W_up: (d_expert, d_internal) each; W_down.T: (d_expert, d_internal)
            W_gate   = torch.stack([e.W_gate.weight for e in exps])   # (m, d_e, D)
            W_up     = torch.stack([e.W_up.weight   for e in exps])   # (m, d_e, D)
            W_down   = torch.stack([e.W_down.weight.T for e in exps]) # (m, d_e, D)
            h_gate   = torch.einsum('bd,med->bme', z, W_gate)         # (B, m, d_e)
            h_up     = torch.einsum('bd,med->bme', z, W_up)           # (B, m, d_e)
            h        = F.silu(h_gate) * h_up                          # (B, m, d_e) SwiGLU
            d_       = torch.einsum('bme,meo->bmo', h, W_down)        # (B, m, D)
            for i, s in enumerate(m_slots):
                all_d[:, pos[s]] = d_[:, i]

        # \u2500\u2500 Sequential PolynomialExperts \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
        for s in self._type_groups.get('poly', []):
            all_d[:, pos[s]] = self.get_expert(s)(z)

        # \u2500\u2500 Sequential TournamentExperts (heterogeneous during grace period) \u2500
        for s in self._type_groups.get('tournament', []):
            all_d[:, pos[s]] = self.get_expert(s)(z)

        return all_d

    # \u2500\u2500 Forward \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    def forward(self, z: torch.Tensor):
        """
        z: (batch, d_internal)
        Returns (z_next, routing_info).
        routing_info includes expert_ids (active slot IDs), topk_idx as positions
        in that list (n_active = null route), z_input for the lifecycle manager,
        all_weights_grad (with gradient) for entropy regularization, and
        null_weight_mean for Comet logging.
        """
        n = self.n_experts
        if n == 0:
            return z, {'weights': None, 'topk_idx': None,
                       'distances': None, 'expert_ids': [], 'z_input': z.detach(),
                       'all_weights_grad': None, 'null_weight_mean': 0.0}

        weights, topk_idx, distances, all_weights_full, null_weight_mean = self.router(z)
        active_slots = self.router.active_slots

        all_deltas = self._batched_dispatch(z, active_slots)  # (B, n, D)

        # Pad with a zero row for the null route (index n contributes zero delta)
        B, D = z.shape
        zero_row    = torch.zeros(B, 1, D, device=z.device, dtype=z.dtype)
        all_deltas_padded = torch.cat([all_deltas, zero_row], dim=1)  # (B, n+1, D)

        idx_exp         = topk_idx.unsqueeze(-1).expand(-1, -1, D)
        selected_deltas = all_deltas_padded.gather(1, idx_exp)
        weighted_delta  = (weights.unsqueeze(-1) * selected_deltas).sum(dim=1)
        z_next          = z + weighted_delta

        routing_info = {
            'weights'          : weights.detach(),
            'topk_idx'         : topk_idx.detach(),
            'distances'        : distances.detach(),
            'expert_ids'       : active_slots,       # position \u2192 global slot ID
            'z_input'          : z.detach(),          # input vectors for lifecycle buffer
            'all_weights_grad' : all_weights_full,    # (B, n+1) with gradient, for entropy
            'null_weight_mean' : null_weight_mean,    # scalar float for Comet logging
        }
        return z_next, routing_info


---
## 4. Expert Lifecycle — Tournament & Manager  *(Phase 3 additions)*

### Marginal Utility Index
Each expert acts as a region proxy in high-D space (its centroid = region centroid).

$$U(e) = \frac{\text{mean\_loss\_credit}(e)}{\log(1 + \text{query\_count}(e))}$$

- **High utility** = high prediction error in a low-query region → spawn a new expert nearby
- Statistics accumulated in a rolling window to track current relevance

### Expert Lifecycle States
```
  [Birth trigger] → TournamentExpert (trial state, grace period)
       ↓ (after grace_period steps)
  promote winner type → live expert
       ↓ (if query_count < death_threshold over rolling window)
  prune → removed from pool
```

### Sequential Tournament
During the grace period the new expert slot cycles through three candidate types:
- Phase 1 of 3: GELU MLP trains for `grace_period // 3` steps
- Phase 2 of 3: SiLU MLP trains for `grace_period // 3` steps  
- Phase 3 of 3: Polynomial expert trains for `grace_period // 3` steps

The candidate with the lowest average LM loss in its phase is promoted.


In [ ]:
class TournamentExpert(nn.Module):
    """
    Sequential tournament expert used during the grace (trial) period.

    Cycles through candidate types: gelu \u2192 silu \u2192 poly.
    Each type trains for grace_period // 3 steps.
    Calling promote() returns the (type_name, module) with the lowest average
    LM loss in its phase.

    Both MLP candidates ('gelu', 'silu') use SwiGLU activation; they differ
    only in random initialisation, providing diversity for the tournament.

    Has the same forward interface as MLPExpert / PolynomialExpert:
        forward(z) \u2192 delta   [shape (batch, d_internal)]
    """

    _PHASES = ['gelu', 'silu', 'poly']

    def __init__(self, d_internal: int, d_expert: int, grace_period: int):
        super().__init__()
        self.grace_period = grace_period
        self._phase_len   = grace_period // len(self._PHASES)

        # Both MLP candidates use SwiGLU; random init gives tournament diversity
        self.candidates = nn.ModuleDict({
            'gelu' : MLPExpert(d_internal, d_expert),
            'silu' : MLPExpert(d_internal, d_expert),
            'poly' : PolynomialExpert(d_internal),
        })

        self._phase_losses = {k: [] for k in self._PHASES}
        self.steps         = 0

    def _current_phase(self) -> str:
        idx = min(self.steps // max(self._phase_len, 1), len(self._PHASES) - 1)
        return self._PHASES[idx]

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """Only the active-phase candidate receives gradient flow."""
        return self.candidates[self._current_phase()](z)

    def record_step_loss(self, lm_loss: float):
        """Call once per training step with the chunk LM loss (detached float)."""
        self._phase_losses[self._current_phase()].append(lm_loss)
        self.steps += 1

    def is_done(self) -> bool:
        return self.steps >= self.grace_period

    def promote(self):
        """Return (winner_key: str, winner_module: nn.Module)."""
        avgs = {
            k: sum(v) / max(len(v), 1)
            for k, v in self._phase_losses.items()
        }
        winner = min(avgs, key=avgs.__getitem__)
        return winner, self.candidates[winner]


In [ ]:
class ExpertLifecycleManager:
    """
    Tracks rolling per-expert statistics and makes split / death decisions.

    Weighted-loss metric:
        Q(e) = sum(w_e(x) * L(x)) / sum(w_e(x))
    High Q → the expert is routed high-loss tokens and not handling them well.

    Birth (split):  split the worst live expert (highest Q among those with
                    enough rolling history). Splits into two TournamentExpert
                    children with centroids offset along a random axis, and
                    output weights distilled via lstsq over the parent's recent
                    input buffer.
    Death:          rolling weight-sum < death_threshold  AND  Q < median Q.
                    (underused AND not even the worst predictor)

    Split cooldown: after any split, no further split until split_cooldown steps.

    DDP-safe: check_lifecycle wraps rank-0 execution and broadcasts events so
    all ranks apply identical pool mutations.

    Not an nn.Module — TournamentExpert children are registered inside the
    processing layer which IS an nn.Module.
    """

    def __init__(
        self,
        grace_period        : int,
        death_threshold     : float,
        rolling_window_size : int,
        d_internal          : int,
        d_expert            : int,
        split_epsilon       : float = 0.3,
        split_cooldown      : int   = 500,
    ):
        self.grace_period        = grace_period
        self.death_threshold     = death_threshold
        self.rolling_window_size = rolling_window_size
        self.d_internal          = d_internal
        self.d_expert            = d_expert
        self.split_epsilon       = split_epsilon
        self.split_cooldown      = split_cooldown

        # Rolling buffers — w*L values and w values per expert
        self._wl_values = defaultdict(
            lambda: deque(maxlen=rolling_window_size))
        self._w_values  = defaultdict(
            lambda: deque(maxlen=rolling_window_size))

        # Per-expert input buffer for neighbor distillation (64 most recent)
        self._input_buffer = defaultdict(lambda: deque(maxlen=64))

        # Lifecycle metadata
        self._state      = {}   # eid → 'trial' | 'live'
        self._birth_step = {}   # eid → global_step
        self._tournaments= {}   # eid → TournamentExpert reference

        # Split cooldown tracker
        self._last_split_step = -split_cooldown

        # Cumulative event counters for logging
        self.n_births = 0
        self.n_deaths = 0

    # ── Registration ──────────────────────────────────────────────────────
    def register_expert(self, eid: int, state: str, birth_step: int,
                        module: nn.Module = None):
        self._state[eid]      = state
        self._birth_step[eid] = birth_step
        if module is not None and isinstance(module, TournamentExpert):
            self._tournaments[eid] = module

    def unregister_expert(self, eid: int):
        for d in (self._state, self._birth_step, self._wl_values,
                  self._w_values, self._input_buffer, self._tournaments):
            d.pop(eid, None)

    # ── Statistics accumulation ────────────────────────────────────────────
    def record_chunk(self, routing_infos_flat: list, per_step_losses: list):
        """
        routing_infos_flat : list of routing_info dicts (one per token×depth step)
        per_step_losses    : list of float LM losses (same length)
        Accumulates w*L and w per expert; stores z_input in per-expert buffer.
        """
        for rinfo, lm_loss in zip(routing_infos_flat, per_step_losses):
            if rinfo.get('topk_idx') is None:
                continue
            expert_ids_list = rinfo['expert_ids']
            topk_idx        = rinfo['topk_idx']    # (batch, top_k)
            weights         = rinfo['weights']      # (batch, top_k)
            z_input         = rinfo.get('z_input')  # (batch, d_internal) or None
            B, K = topk_idx.shape
            for b in range(B):
                for k in range(K):
                    pos = topk_idx[b, k].item()
                    if pos < len(expert_ids_list):
                        eid = expert_ids_list[pos]
                        w   = weights[b, k].item()
                        self._wl_values[eid].append(w * lm_loss)
                        self._w_values[eid].append(w)
                        if z_input is not None:
                            self._input_buffer[eid].append(
                                z_input[b].cpu())

    # ── Q(e) computation ──────────────────────────────────────────────────
    def _compute_Q(self, eid: int) -> float:
        """Weighted average loss routed to expert e over the rolling window."""
        w_sum  = sum(self._w_values[eid])
        if w_sum < 1e-8:
            return 0.0
        return sum(self._wl_values[eid]) / w_sum

    # ── Neighbor distillation ──────────────────────────────────────────────
    def _distill_child(self, child: 'TournamentExpert', parent: nn.Module,
                       input_buffer: list, dev):
        """Initialise child candidate output projections to approximate parent.
        Solves lstsq(hidden_features, parent_outputs) per candidate type."""
        if len(input_buffer) < 8:
            return
        X = torch.stack(input_buffer).to(dev)   # (n, d_internal)
        with torch.no_grad():
            Y = parent(X)                        # (n, d_internal) — parent delta
        for key, cand in child.candidates.items():
            with torch.no_grad():
                if isinstance(cand, MLPExpert):
                    # SwiGLU: compute gated activation, then solve for W_down
                    z_n    = cand.norm(X)
                    gate   = F.silu(z_n @ cand.W_gate.weight.T)  # (n, d_expert)
                    up     = z_n @ cand.W_up.weight.T             # (n, d_expert)
                    h      = gate * up                            # SwiGLU output
                    result = torch.linalg.lstsq(h, Y)
                    # solution: (d_expert, d_internal) ≈ W_down.T
                    cand.W_down.weight.copy_(result.solution.T)
                elif isinstance(cand, PolynomialExpert):
                    z_n    = cand.norm(X)
                    feat   = torch.cat([z_n, z_n.pow(2)], dim=-1)  # (n, 2*D)
                    result = torch.linalg.lstsq(feat, Y)
                    cand.proj.weight.copy_(result.solution.T)

    # ── Split worst expert ─────────────────────────────────────────────────
    def _split_worst(self, global_step: int, processing_layer,
                     dev) -> list:
        """
        Identify the live expert with the highest Q(e) that has accumulated
        at least rolling_window_size/4 weight-sum entries, then:
          1. Delete that expert from the pool.
          2. Spawn two TournamentExpert children with centroids offset ±ε along
             a random unit axis and output weights distilled from the parent.
        Returns list of event strings (empty if no eligible expert found).
        """
        min_entries = self.rolling_window_size // 4
        candidates  = {}
        for eid, state in self._state.items():
            if state != 'live':
                continue
            if len(self._w_values[eid]) < min_entries:
                continue
            candidates[eid] = self._compute_Q(eid)

        if not candidates:
            return []

        worst_eid       = max(candidates, key=candidates.__getitem__)
        parent_centroid = processing_layer.router.centroids.data[
            worst_eid].clone().to(dev)
        parent_expert   = processing_layer.get_expert(worst_eid)
        input_buf       = list(self._input_buffer[worst_eid])

        # Centroid split along random unit axis
        axis = F.normalize(torch.randn(self.d_internal, device=dev), dim=0)
        eps  = self.split_epsilon
        c1   = F.normalize(parent_centroid + eps * axis, dim=0)
        c2   = F.normalize(parent_centroid - eps * axis, dim=0)

        # Create TournamentExpert children
        child1 = TournamentExpert(self.d_internal, self.d_expert,
                                  self.grace_period).to(dev)
        child2 = TournamentExpert(self.d_internal, self.d_expert,
                                  self.grace_period).to(dev)

        # Neighbor distillation — initialise output projections from parent
        if len(input_buf) >= 8:
            self._distill_child(child1, parent_expert, input_buf, dev)
            self._distill_child(child2, parent_expert, input_buf, dev)

        # Remove parent, register children
        processing_layer.remove_expert(worst_eid)
        self.unregister_expert(worst_eid)

        eid1 = processing_layer.add_expert(
            child1, c1, label=f'trial_{global_step}_a')
        eid2 = processing_layer.add_expert(
            child2, c2, label=f'trial_{global_step}_b')

        self.register_expert(eid1, state='trial',
                             birth_step=global_step, module=child1)
        self.register_expert(eid2, state='trial',
                             birth_step=global_step, module=child2)

        self._last_split_step = global_step
        self.n_births         += 1

        return [
            f'[step {global_step}] Expert {worst_eid} split → '
            f'children {eid1},{eid2} '
            f'(Q={candidates[worst_eid]:.4f}, pool={processing_layer.n_experts})'
        ]

    # ── Lifecycle check ────────────────────────────────────────────────────
    def check_lifecycle(
        self,
        global_step      : int,
        processing_layer,           # MoEProcessingLayer
        max_experts      : int,
        dev              = None,    # torch.device for new experts
    ) -> list:
        """
        Evaluates split and death conditions. Returns list of event strings.
        Call this periodically (every lifecycle_check_interval steps).
        In DDP training, call this only on rank-0 and broadcast results.
        """
        events = []
        if dev is None:
            dev = next(iter(processing_layer.parameters())).device

        # ── 1. Promote completed tournaments ──────────────────────────────
        for eid in list(self._tournaments.keys()):
            t = self._tournaments[eid]
            if t.is_done():
                winner_key, winner_module = t.promote()
                processing_layer.replace_expert(
                    eid, winner_module.to(dev),
                    new_label=f'{winner_key}_{eid}'
                )
                del self._tournaments[eid]
                self._state[eid] = 'live'
                events.append(
                    f'[step {global_step}] Expert {eid}: '
                    f'tournament → {winner_key} promoted'
                )

        # ── 2. Death check (dual condition: low weight sum AND low Q) ────
        live_Q = {
            eid: self._compute_Q(eid)
            for eid, state in self._state.items() if state == 'live'
        }
        q_sorted = sorted(live_Q.values())
        median_Q = q_sorted[len(q_sorted) // 2] if q_sorted else 0.0

        for eid in list(self._state.keys()):
            if self._state[eid] != 'live':
                continue
            w_sum = sum(self._w_values.get(eid, deque()))
            q_val = live_Q.get(eid, 0.0)
            if w_sum < self.death_threshold and q_val < median_Q:
                processing_layer.remove_expert(eid)
                self.unregister_expert(eid)
                self.n_deaths += 1
                events.append(
                    f'[step {global_step}] Expert {eid}: pruned '
                    f'(w_sum={w_sum:.3f}, Q={q_val:.4f})'
                )

        # ── 3. Split check (replaces birth check) ─────────────────────────
        steps_since_split = global_step - self._last_split_step
        if (steps_since_split >= self.split_cooldown
                and processing_layer.n_experts < max_experts):
            events += self._split_worst(global_step, processing_layer, dev)

        return events


---
## 5. Input / Gate / Output Layers  *(unchanged from Phase 2)*


In [ ]:
class InputLayer(nn.Module):
    """
    Reads SSM state h_{t-1} and token embedding e_t.
    Projects them into the internal processing space z_0.
    Executed exactly once per token.
    """
    def __init__(self, d_embed: int, d_state: int, d_internal: int):
        super().__init__()
        self.proj = nn.Linear(d_embed + d_state, d_internal, bias=False)
        self.norm = RMSNorm(d_internal)

    def forward(self, e_t: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        return self.norm(self.proj(torch.cat([e_t, h], dim=-1)))


class GateMLP(nn.Module):
    """
    Depth gate: evaluates whether z_k requires further processing.
    Produces a scalar score s ~ N(0,1) (enforced by KS anchoring loss).
    P(go deeper) = \u03a6(s)  \u2014  standard normal CDF.
    """
    def __init__(self, d_internal: int):
        super().__init__()
        self.net = nn.Sequential(
            RMSNorm(d_internal),
            nn.Linear(d_internal, d_internal // 2, bias=False),
            nn.GELU(),
            nn.Linear(d_internal // 2, 1, bias=False)
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).squeeze(-1)

    @staticmethod
    def normal_cdf(s: torch.Tensor) -> torch.Tensor:
        return 0.5 * (1.0 + torch.erf(s / math.sqrt(2.0)))


class OutputLayer(nn.Module):
    """
    Projects internal representation to vocabulary logits and updates SSM state.
    Executed exactly once per token at the final processing depth z_K.

    Weight tying: logits are computed as to_embed(norm(z)) @ embedding.weight.T,
    sharing the input embedding matrix (vocab_size \u00d7 d_embed).
    to_embed is a learned (d_internal \u2192 d_embed) linear that bridges the two
    spaces; it is the only new parameter introduced by tying.
    The embedding reference is stored as a plain Python attribute (not an
    nn.Module submodule) to avoid double-counting parameters.

    The SSM state update path (to_state via W_gate + W_candidate) is unchanged.
    """
    def __init__(self, d_internal: int, d_state: int, vocab_size: int,
                 d_embed: int, embedding: nn.Embedding):
        super().__init__()
        self.norm        = RMSNorm(d_internal)
        self.to_embed    = nn.Linear(d_internal, d_embed, bias=False)
        self.W_gate      = nn.Linear(d_internal + d_state, d_state, bias=False)
        self.W_candidate = nn.Linear(d_internal, d_state, bias=False)
        # Shared reference \u2014 bypass nn.Module.__setattr__ to avoid double-registering
        # the embedding parameters. Plain Python attribute, not an nn.Module child.
        self.__dict__['_embedding'] = embedding

    def get_logits(self, z: torch.Tensor) -> torch.Tensor:
        z_embed = self.to_embed(self.norm(z))          # (batch, d_embed)
        return z_embed @ self._embedding.weight.T      # (batch, vocab_size)

    def update_state(self, z: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        gate      = torch.sigmoid(self.W_gate(torch.cat([z, h], dim=-1)))
        candidate = torch.tanh(self.W_candidate(z))
        return gate * h + (1.0 - gate) * candidate


---
## 6. Full Model — AdaptiveSSMMoE  *(updated _init_weights for dynamic pool)*

Phase 3 change: `n_experts_init` replaces `n_experts` in constructor.
`_init_weights` iterates `get_all_experts()` and handles `TournamentExpert` candidates.


In [ ]:
class AdaptiveSSMMoE(nn.Module):
    """
    Adaptive Compute SSM \u2014 Phase 3 (dynamic MoE processing layer).

    Architecture (strictly separated roles):
      InputLayer         : (e_t, h_{t-1}) \u2192 z_0              [once per token]
      GateMLP            : z_k \u2192 scalar s_k                   [at each depth]
      MoEProcessingLayer : z_k \u2192 z_{k+1}  (shared weights)   [0..D times, dynamic pool]
      OutputLayer        : z_K \u2192 logits + h_t                  [once per token]
    """

    def __init__(
        self,
        vocab_size      : int,
        d_embed         : int,
        d_state         : int,
        d_internal      : int,
        d_expert        : int,
        n_experts_init  : int,
        top_k           : int,
        max_depth       : int   = 1,
        temperature     : float = 1.0,
        max_experts     : int   = 64,
        faiss_threshold : int   = 32,
    ):
        super().__init__()
        self.d_state   = d_state
        self.max_depth = max_depth

        self.embedding        = nn.Embedding(vocab_size, d_embed)
        self.input_layer      = InputLayer(d_embed, d_state, d_internal)
        self.gate             = GateMLP(d_internal)
        self.processing_layer = MoEProcessingLayer(
            d_internal, d_expert, n_experts_init, top_k, temperature,
            max_experts=max_experts, faiss_threshold=faiss_threshold
        )
        # OutputLayer receives d_embed and the embedding reference for weight tying
        self.output_layer     = OutputLayer(
            d_internal, d_state, vocab_size, d_embed, self.embedding
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
        # Re-zero expert output projections (xavier above would overwrite)
        for exp in self.processing_layer.get_all_experts():
            self._zero_expert_outputs(exp)

    @staticmethod
    def _zero_expert_outputs(exp):
        """Zero-init output projection of a single expert or tournament group."""
        if isinstance(exp, MLPExpert):
            nn.init.zeros_(exp.W_down.weight)
        elif isinstance(exp, PolynomialExpert):
            nn.init.zeros_(exp.proj.weight)
        elif isinstance(exp, TournamentExpert):
            for cand in exp.candidates.values():
                AdaptiveSSMMoE._zero_expert_outputs(cand)

    def get_ssm_transition_params(self) -> list:
        """Return SSM state transition parameters for dedicated Muon treatment.
        W_gate and W_candidate implement h_t = gate * h_{t-1} + (1-gate) * candidate,
        analogous to the A matrix in a linear SSM."""
        return (list(self.output_layer.W_gate.parameters())
                + list(self.output_layer.W_candidate.parameters()))

    def init_state(self, batch_size: int, device) -> torch.Tensor:
        return torch.zeros(batch_size, self.d_state, device=device)

    # \u2500\u2500 Training: full-unroll forward for one token \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    def forward_token(self, token_ids, h, max_depth):
        e_t = self.embedding(token_ids)
        z   = self.input_layer(e_t, h)

        all_logits    = []
        gate_scores   = []
        routing_infos = []

        for _ in range(max_depth):
            s_k = self.gate(z)
            gate_scores.append(s_k)
            all_logits.append(self.output_layer.get_logits(z))
            z, rinfo = self.processing_layer(z)
            routing_infos.append(rinfo)

        all_logits.append(self.output_layer.get_logits(z))
        h_new = self.output_layer.update_state(z, h)

        return all_logits, gate_scores, h_new, routing_infos

    # \u2500\u2500 Inference: early-exit with Smirnov depth control \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    @torch.no_grad()
    def generate(
        self,
        prompt_ids     : torch.Tensor,
        max_new_tokens : int   = 100,
        mu             : float = 0.0,
        temperature    : float = 1.0,
        top_k          : int   = 50,
    ) -> torch.Tensor:
        self.eval()
        dev = next(self.parameters()).device
        h   = self.init_state(1, dev)
        generated = prompt_ids.to(dev)

        for t in range(generated.shape[1] - 1):
            _, _, h, _ = self.forward_token(generated[:, t], h, self.max_depth)

        for _ in range(max_new_tokens):
            e_t = self.embedding(generated[:, -1])
            z   = self.input_layer(e_t, h)
            for _ in range(self.max_depth):
                s = self.gate(z)
                if torch.rand(1, device=dev).item() > GateMLP.normal_cdf(s + mu).mean().item():
                    break
                z, _ = self.processing_layer(z)
            logits = self.output_layer.get_logits(z) / temperature
            h      = self.output_layer.update_state(z, h)
            if top_k > 0:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1:]] = float('-inf')
            next_tok  = torch.multinomial(F.softmax(logits, dim=-1), 1)
            generated = torch.cat([generated, next_tok], dim=1)

        return generated


---
## 7. Training Objective  *(unchanged except centroid repulsion guard for n < 2)*

$$L = L_{\text{LM}} + \lambda_{\text{KS}} \cdot D(p_s \| \mathcal{N}(0,1)) + \lambda_{\text{norm}} \cdot (\ln \|h_t\|)^2 + \lambda_{\text{rep}} \cdot L_{\text{repulsion}}$$

`centroid_repulsion_loss` now guards against pools with fewer than 2 experts.


In [ ]:
def compute_lm_loss(all_logits, gate_scores, targets, max_depth):
    device = targets.device
    batch  = targets.shape[0]
    p_continue = [GateMLP.normal_cdf(s) for s in gate_scores]
    p_reach    = torch.ones(batch, device=device)
    total_loss = torch.zeros(1, device=device)
    for k in range(max_depth):
        p_exit     = p_reach * (1.0 - p_continue[k])
        ce_k       = F.cross_entropy(all_logits[k], targets, reduction='none')
        total_loss = total_loss + (p_exit * ce_k).mean()
        p_reach    = p_reach * p_continue[k]
    ce_final   = F.cross_entropy(all_logits[max_depth], targets, reduction='none')
    total_loss = total_loss + (p_reach * ce_final).mean()
    return total_loss


def ks_anchoring_loss(gate_scores_list):
    """Return (mu_loss, sigma_loss) independently.
    mu_loss    = gate_scores.mean() ** 2          (penalise mean ≠ 0)
    sigma_loss = (gate_scores.std() - 1.0) ** 2  (penalise std ≠ 1)
    Each term is weighted separately via lambda_ks_mu / lambda_ks_sigma
    in the training loop.
    """
    scores     = torch.cat([s.flatten() for s in gate_scores_list])
    mu_loss    = scores.mean() ** 2
    sigma_loss = (scores.std() - 1.0) ** 2
    return mu_loss, sigma_loss


def norm_regularization_loss(h):
    norms = torch.norm(h, dim=-1)
    return torch.mean(torch.log(norms + 1e-8) ** 2)


def centroid_repulsion_loss(centroids, margin=2.0):
    """Guard: returns 0 if fewer than 2 experts (pool growing from small init)."""
    n = centroids.shape[0]
    if n < 2:
        return torch.tensor(0.0, device=centroids.device, requires_grad=False)
    diff = centroids.unsqueeze(0) - centroids.unsqueeze(1)
    dist = diff.norm(dim=-1)
    mask = (1.0 - torch.eye(n, device=centroids.device))
    rep  = (torch.clamp(margin - dist, min=0.0) ** 2) * mask
    return rep.sum() / (n * (n - 1))


---
## 8. Dataset — TinyStories  *(unchanged from Phase 2)*


In [ ]:
class TinyStoriesDataset(Dataset):
    """
    Streams TinyStories from HuggingFace.
    Returns (input_ids, target_ids) pairs of fixed length seq_len.
    """
    def __init__(self, split: str, seq_len: int, max_stories=None):
        self.seq_len  = seq_len
        tokenizer     = AutoTokenizer.from_pretrained('gpt2')
        tokenizer.pad_token = tokenizer.eos_token
        ds = load_dataset('roneneldan/TinyStories', split=split, streaming=False)
        if max_stories:
            ds = ds.select(range(min(max_stories, len(ds))))
        self.samples = []
        for row in tqdm(ds, desc=f'Tokenising {split}', leave=False):
            ids = tokenizer.encode(row['text'], truncation=True, max_length=seq_len + 1)
            if len(ids) < 2:
                continue
            if len(ids) < seq_len + 1:
                ids = ids + [tokenizer.eos_token_id] * (seq_len + 1 - len(ids))
            ids = ids[:seq_len + 1]
            self.samples.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        return ids[:-1], ids[1:]


---
## 9. Training Loop — TBPTT + Lifecycle Hooks

Phase 3 additions:
- `build_optimizers(model, config, current_step, total_steps)` — standalone helper, called
  on startup and after each pool-change event to refresh parameter groups
- Per-chunk: collect `lifecycle_routing_infos` + `lifecycle_losses` for the lifecycle manager
- Per trial expert: call `tournament.record_step_loss(lm_loss)` to advance tournament
- Every `lifecycle_check_interval` steps: `lifecycle_manager.check_lifecycle(...)` and
  rebuild optimizers if any events occurred
- New CometML metrics: `expert_pool_size`, `expert_births`, `expert_deaths`,
  `marginal_utility_max`, `tournament_winner`


In [ ]:
def build_optimizers(model, config, total_steps: int = None):
    """
    Build Muon + AdamW + Muon-SSM optimizers for the model.
    Called once at the start of training \u2014 the expert pool is fixed-size so
    the parameter set never changes and this function is never re-called.

    Three parameter groups:
      optimizer      \u2014 Muon for all 2-D non-embedding, non-SSM parameters
      optimizer_adamw\u2014 AdamW for 1-D params and embedding
      optimizer_ssm  \u2014 Muon at lr*0.5 for SSM state transition matrices
                          (W_gate and W_candidate in OutputLayer), which play
                          the role of the recurrent A matrix in linear SSMs

    Returns (optimizer, optimizer_adamw, optimizer_ssm,
             scheduler, scheduler_adamw, scheduler_ssm).
    """
    emb_ids  = {id(p) for p in model.embedding.parameters()}
    ssm_ids  = {id(p) for p in model.get_ssm_transition_params()}

    muon_params  = [p for p in model.parameters()
                    if p.ndim >= 2 and id(p) not in emb_ids and id(p) not in ssm_ids]
    adamw_params = [p for p in model.parameters()
                    if p.ndim < 2  or  id(p) in emb_ids]
    ssm_params   = model.get_ssm_transition_params()

    optimizer       = torch.optim.Muon(muon_params, lr=config['lr'])
    optimizer_adamw = torch.optim.AdamW(
        adamw_params, lr=config['lr'],
        weight_decay=config['weight_decay'], betas=(0.9, 0.95)
    )
    optimizer_ssm   = torch.optim.Muon(ssm_params, lr=config['lr'] * 0.5)

    t_max = max(total_steps or 1, 1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=t_max, eta_min=config['lr'] * 0.1
    )
    scheduler_adamw = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_adamw, T_max=t_max, eta_min=config['lr'] * 0.1
    )
    scheduler_ssm = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_ssm, T_max=t_max, eta_min=config['lr'] * 0.05
    )
    return optimizer, optimizer_adamw, optimizer_ssm, scheduler, scheduler_adamw, scheduler_ssm


In [ ]:
def train(config: dict, experiment=None) -> nn.Module:
    import glob, re

    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    if experiment:
        experiment.log_parameters(config)

    tokenizer = AutoTokenizer.from_pretrained(config['tokenizer'])
    tokenizer.pad_token = tokenizer.eos_token
    vocab_size = tokenizer.vocab_size

    # ── Checkpoint resume ──────────────────────────────────────────────────
    # Hard compat: architecture dimensions — changing these requires re-init.
    _HARD_COMPAT = {'d_embed', 'd_state', 'd_internal', 'd_expert', 'tokenizer'}
    # Soft compat: top_k is a runtime attribute (applied post-load);
    #              max_experts controls pool ceiling (handled during build).

    resume_ckpt = None
    ckpt_files  = sorted(
        glob.glob('adaptive_ssm_moe_p3_epoch*.pt'),
        key=lambda f: int(re.search(r'epoch(\d+)', f).group(1))
    )
    for cf in reversed(ckpt_files):          # latest epoch first
        try:
            candidate = torch.load(cf, map_location='cpu')
            saved_cfg = candidate.get('config', {})
            mismatched_hard = [k for k in _HARD_COMPAT
                               if saved_cfg.get(k) != config.get(k)]
            if mismatched_hard:
                print(f'Skipping incompatible checkpoint {cf} '
                      f'(mismatched keys: {mismatched_hard})')
                continue
            resume_ckpt = candidate
            print(f'Resuming from checkpoint: {cf}  '
                  f'(epoch {resume_ckpt["epoch"]}, '
                  f'val_loss={resume_ckpt["val_loss"]:.4f})')
            for k in ('top_k', 'max_experts'):
                sv, cv = saved_cfg.get(k), config.get(k)
                if sv != cv:
                    print(f'  NOTE: {k} changed {sv} → {cv} (will be applied after load)')
            break
        except Exception as e:
            print(f'Could not load {cf}: {e}')

    if not ckpt_files:
        print('No existing checkpoints found — starting from scratch.')

    # ── Build model ────────────────────────────────────────────────────────
    # When resuming, build with the saved max_experts so centroids / expert_active
    # tensor shapes match the state_dict exactly.  top_k is just a Python
    # attribute and is overridden post-load; pool resize happens post-load too.
    n_experts_init = config['n_experts_init']
    build_max_experts = config['max_experts']
    if resume_ckpt is not None:
        _sc = resume_ckpt.get('config', {})
        ckpt_max = _sc.get('max_experts', config['max_experts'])
        if ckpt_max != config['max_experts']:
            _active_mask = resume_ckpt['model_state_dict'].get(
                'processing_layer.router.expert_active')
            _n_active = int(_active_mask.sum().item()) if _active_mask is not None else 0
            if config['max_experts'] < _n_active:
                print(f'  WARNING: new max_experts={config["max_experts"]} < '
                      f'{_n_active} active experts in checkpoint — '
                      f'using saved value {ckpt_max}.')
            build_max_experts = ckpt_max   # always load with saved size; resize post-load

    model = AdaptiveSSMMoE(
        vocab_size      = vocab_size,
        d_embed         = config['d_embed'],
        d_state         = config['d_state'],
        d_internal      = config['d_internal'],
        d_expert        = config['d_expert'],
        n_experts_init  = n_experts_init,
        top_k           = config['top_k'],
        max_depth       = config['max_depth'],
        temperature     = config['router_temperature'],
        max_experts     = build_max_experts,
        faiss_threshold = config['faiss_threshold'],
    ).to(dev)

    if resume_ckpt is not None:
        try:
            saved_sd = resume_ckpt['model_state_dict']

            # ── Reconstruct expert types to match checkpoint ────────────────
            # Parse state_dict keys to infer each slot's type, then replace
            # mismatched modules before load_state_dict so the parameter
            # shapes align (avoids TournamentExpert vs MLPExpert mismatch).
            import re as _re
            slot_types: dict = {}  # slot -> 'tournament' | 'poly' | 'mlp'
            for key in saved_sd:
                m = _re.match(r'processing_layer\.expert_(\d+)\.(candidates|proj|W_gate)', key)
                if m:
                    slot = int(m.group(1))
                    sig  = m.group(2)
                    if sig == 'candidates':
                        slot_types[slot] = 'tournament'
                    elif sig == 'proj':
                        slot_types.setdefault(slot, 'poly')
                    else:  # W_gate
                        slot_types.setdefault(slot, 'mlp')

            layer = model.processing_layer
            for slot, stype in slot_types.items():
                cur = layer.get_expert(slot)
                if stype == 'tournament' and not isinstance(cur, TournamentExpert):
                    layer.register_module(
                        f'expert_{slot}',
                        TournamentExpert(config['d_internal'], config['d_expert'],
                                         config['grace_period'])
                    )
                elif stype == 'poly' and not isinstance(cur, PolynomialExpert):
                    layer.register_module(f'expert_{slot}', PolynomialExpert(config['d_internal']))
                elif stype == 'mlp' and not isinstance(cur, MLPExpert):
                    layer.register_module(
                        f'expert_{slot}',
                        MLPExpert(config['d_internal'], config['d_expert'])
                    )
            layer._rebuild_type_groups()
            # ───────────────────────────────────────────────────────────────

            model.load_state_dict(saved_sd)
            model.processing_layer.expert_labels = resume_ckpt['expert_labels']
            print(f'Expert pool restored ({model.processing_layer.n_experts}): '
                  f'{[l for l in model.processing_layer.expert_labels if l]}')

            # ── Apply soft-compat overrides ─────────────────────────────────
            # top_k: just update the router attribute — no tensor shapes change.
            _sc = resume_ckpt.get('config', {})
            if _sc.get('top_k') != config['top_k']:
                model.processing_layer.router.top_k = config['top_k']
                print(f'  top_k set to {config["top_k"]}')

            # max_experts: resize centroids, expert_active, and expert modules.
            _router = model.processing_layer.router
            _layer  = model.processing_layer
            _old_max = _router.max_experts
            _new_max = config['max_experts']
            if _new_max > _old_max:
                # Expand: add inactive slots; existing active experts are unaffected.
                with torch.no_grad():
                    _old_c = _router.centroids.data
                    _new_c = torch.zeros(_new_max, config['d_internal'],
                                         device=_old_c.device, dtype=_old_c.dtype)
                    _new_c[:_old_max] = _old_c
                    _router.centroids = nn.Parameter(_new_c)
                    _new_a = torch.zeros(_new_max, dtype=torch.bool, device=_old_c.device)
                    _new_a[:_old_max] = _router.expert_active
                    _router.register_buffer('expert_active', _new_a)
                _router.max_experts = _new_max
                for _i in range(_old_max, _new_max):
                    _t = _layer._TYPE_CYCLE[_i % len(_layer._TYPE_CYCLE)]
                    _layer.register_module(f'expert_{_i}', _layer._make_expert(_t))
                _layer.max_experts = _new_max
                _layer.expert_labels += [''] * (_new_max - _old_max)
                _layer._rebuild_type_groups()
                print(f'  max_experts expanded: {_old_max} → {_new_max}')
            elif _new_max < _old_max:
                # Shrink: only safe when no active slots have index >= _new_max.
                if _router.expert_active[_new_max:].any().item():
                    print(f'  WARNING: cannot shrink max_experts to {_new_max} — '
                          f'active experts exist at slots >= {_new_max}. '
                          f'Keeping {_old_max}.')
                else:
                    with torch.no_grad():
                        _router.centroids = nn.Parameter(
                            _router.centroids.data[:_new_max])
                        _router.register_buffer('expert_active',
                                                _router.expert_active[:_new_max])
                    _router.max_experts = _new_max
                    for _i in range(_new_max, _old_max):
                        _layer._modules.pop(f'expert_{_i}', None)
                    _layer.max_experts = _new_max
                    _layer.expert_labels = _layer.expert_labels[:_new_max]
                    _layer._rebuild_type_groups()
                    print(f'  max_experts shrunk: {_old_max} → {_new_max}')
            # ────────────────────────────────────────────────────────────────
        except Exception as e:
            print(f'WARNING: state_dict load failed ({e}). Starting from scratch.')
            resume_ckpt = None
            # Re-initialise with the original expert count
            model = AdaptiveSSMMoE(
                vocab_size      = vocab_size,
                d_embed         = config['d_embed'],
                d_state         = config['d_state'],
                d_internal      = config['d_internal'],
                d_expert        = config['d_expert'],
                n_experts_init  = config['n_experts_init'],
                top_k           = config['top_k'],
                max_depth       = config['max_depth'],
                temperature     = config['router_temperature'],
                max_experts     = config['max_experts'],
                faiss_threshold = config['faiss_threshold'],
            ).to(dev)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'Model parameters: {n_params:,}')
    if resume_ckpt is None:
        print(f'Initial expert pool ({model.processing_layer.n_experts}): '
              f'{model.processing_layer.expert_labels}')
    if experiment:
        experiment.log_parameter('n_params', n_params)

    # ── Register experts with lifecycle manager ────────────────────────────
    lifecycle_manager = ExpertLifecycleManager(
        grace_period        = config['grace_period'],
        death_threshold     = config['death_threshold'],
        rolling_window_size = config['rolling_window_size'],
        d_internal          = config['d_internal'],
        d_expert            = config['d_expert'],
        split_epsilon       = config['split_epsilon'],
        split_cooldown      = config['split_cooldown'],
    )
    for eid in model.processing_layer._expert_ids:
        lifecycle_manager.register_expert(eid, state='live', birth_step=0)
    if resume_ckpt is not None:
        lifecycle_manager.n_births = resume_ckpt['lifecycle_n_births']
        lifecycle_manager.n_deaths = resume_ckpt['lifecycle_n_deaths']

    train_ds = TinyStoriesDataset('train',      config['seq_len'], config.get('num_train'))
    val_ds   = TinyStoriesDataset('validation', config['seq_len'], config.get('num_val'))
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'],
                              shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=config['batch_size'],
                              shuffle=False, num_workers=2, pin_memory=True)

    steps_per_epoch = len(train_loader) * (config['seq_len'] // config['chunk_size'])
    total_steps     = config['epochs'] * steps_per_epoch

    start_epoch = resume_ckpt['epoch'] if resume_ckpt is not None else 0
    global_step = start_epoch * steps_per_epoch

    optimizer, optimizer_adamw, optimizer_ssm, scheduler, scheduler_adamw, scheduler_ssm = build_optimizers(
        model, config, total_steps=total_steps
    )
    scaler = GradScaler('cuda')

    for epoch in range(start_epoch, config['epochs']):
        model.train()
        epoch_lm = epoch_ks_mu = epoch_ks_sigma = epoch_norm = epoch_rep = n_chunks = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["epochs"]}')

        for input_ids, target_ids in pbar:
            input_ids  = input_ids.to(dev)
            target_ids = target_ids.to(dev)
            batch_size = input_ids.shape[0]
            seq_len    = input_ids.shape[1]

            h = model.init_state(batch_size, dev)

            for chunk_start in range(0, seq_len, config['chunk_size']):
                chunk_end = min(chunk_start + config['chunk_size'], seq_len)

                optimizer.zero_grad()
                optimizer_adamw.zero_grad()
                optimizer_ssm.zero_grad()

                chunk_lm   = torch.zeros(1, device=dev)
                chunk_norm = torch.zeros(1, device=dev)
                chunk_ent  = torch.zeros(1, device=dev)
                all_gate_scores = []

                # Lifecycle accumulation buffers (detached, no RAM overhead)
                lifecycle_routing_infos = []
                lifecycle_losses        = []

                # Expert utilisation counter
                n_experts_now = model.processing_layer.n_experts
                expert_counts = {eid: 0 for eid in model.processing_layer._expert_ids}
                dist_sum = weight_ent = null_w_sum = 0.0
                n_tok = 0

                with autocast('cuda'):
                    for t in range(chunk_start, chunk_end):
                        all_logits, gate_scores, h_new, routing_infos = model.forward_token(
                            input_ids[:, t], h, config['max_depth']
                        )
                        lm_t   = compute_lm_loss(
                            all_logits, gate_scores, target_ids[:, t], config['max_depth']
                        )
                        norm_t = norm_regularization_loss(h_new)

                        chunk_lm   = chunk_lm   + lm_t
                        chunk_norm = chunk_norm + norm_t
                        all_gate_scores.extend(gate_scores)

                        # ── Lifecycle: accumulate per routing step ──────────
                        lm_t_val = lm_t.detach().item()
                        for rinfo in routing_infos:
                            lifecycle_routing_infos.append(rinfo)
                            lifecycle_losses.append(lm_t_val)
                            # Expert utilisation
                            if rinfo['topk_idx'] is not None:
                                for pos in rinfo['topk_idx'].view(-1).tolist():
                                    if pos < len(rinfo['expert_ids']):
                                        eid = rinfo['expert_ids'][pos]
                                        expert_counts[eid] = expert_counts.get(eid, 0) + 1
                                dist_sum   += rinfo['distances'].min(dim=-1).values.mean().item()
                                w = rinfo['weights'].clamp(min=1e-8)
                                weight_ent += -(w * w.log()).sum(dim=-1).mean().item()

                        # ── Advance tournament step counters ────────────────
                        for tour in lifecycle_manager._tournaments.values():
                            tour.record_step_loss(lm_t_val)

                        # \u2500\u2500 Entropy regularization accumulation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
                        for rinfo in routing_infos:
                            aw = rinfo.get('all_weights_grad')
                            if aw is not None:
                                chunk_ent = chunk_ent + (
                                    -(aw * aw.clamp(min=1e-8).log()).sum(dim=-1).mean()
                                )
                            null_w_sum += rinfo.get('null_weight_mean', 0.0)

                        h     = h_new
                        n_tok += 1

                    chunk_lm   = chunk_lm   / n_tok
                    chunk_norm = chunk_norm / n_tok
                    chunk_ent  = chunk_ent  / n_tok

                    _active_s  = model.processing_layer.router.active_slots
                    _act_t     = torch.tensor(_active_s, device=dev, dtype=torch.long)
                    centroids  = model.processing_layer.router.centroids[_act_t]
                    rep_loss   = centroid_repulsion_loss(centroids, margin=config['repulsion_margin'])

                    ks_mu_loss, ks_sigma_loss = ks_anchoring_loss(all_gate_scores)
                    lambda_ent = (config['lambda_ent_max']
                                  * min(1.0, global_step / max(config['ent_warmup_steps'], 1)))
                    loss = (
                        chunk_lm
                        + config['lambda_ks_mu']   * ks_mu_loss
                        + config['lambda_ks_sigma'] * ks_sigma_loss
                        + config['lambda_norm']     * chunk_norm
                        + config['lambda_rep']      * rep_loss
                        + lambda_ent               * chunk_ent
                    )

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                scaler.unscale_(optimizer_adamw)
                scaler.unscale_(optimizer_ssm)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
                scaler.step(optimizer)
                scaler.step(optimizer_adamw)
                scaler.step(optimizer_ssm)
                scaler.update()
                scheduler.step()
                scheduler_adamw.step()
                scheduler_ssm.step()

                h = h.detach()

                # ── Record chunk stats in lifecycle manager ─────────────────
                lifecycle_manager.record_chunk(lifecycle_routing_infos, lifecycle_losses)

                # ── Periodic FAISS index rebuild ─────────────────────────────
                if global_step % config['faiss_rebuild_interval'] == 0 and global_step > 0:
                    model.processing_layer.router.rebuild_faiss_index()

                # ── Periodic lifecycle check ────────────────────────────────
                if global_step % config['lifecycle_check_interval'] == 0 and global_step > 0:
                    # DDP-safe: run on rank-0 only, broadcast events to all ranks
                    if dist.is_initialized():
                        if dist.get_rank() == 0:
                            _evts = lifecycle_manager.check_lifecycle(
                                global_step, model.processing_layer,
                                config['max_experts'], dev=dev,
                            )
                        else:
                            _evts = []
                        _ec = [_evts]
                        dist.broadcast_object_list(_ec, src=0)
                        events = _ec[0]
                    else:
                        events = lifecycle_manager.check_lifecycle(
                            global_step, model.processing_layer,
                            config['max_experts'], dev=dev,
                        )
                    if events:
                        print(f'  Pool event(s) at step {global_step}:')
                        for ev in events:
                            print(f'    {ev}')
                            if experiment:
                                experiment.log_text(ev, step=global_step)

                    if experiment:
                        experiment.log_metric('expert_pool_size',
                                              model.processing_layer.n_experts, step=global_step)
                        _q_vals = {
                            eid: lifecycle_manager._compute_Q(eid)
                            for eid, st in lifecycle_manager._state.items()
                            if st == 'live'
                        }
                        if _q_vals:
                            experiment.log_metric('Q_max',
                                                  max(_q_vals.values()), step=global_step)
                        experiment.log_metric('expert_births',
                                              lifecycle_manager.n_births, step=global_step)
                        experiment.log_metric('expert_deaths',
                                              lifecycle_manager.n_deaths, step=global_step)

                epoch_lm       += chunk_lm.item()
                epoch_ks_mu    += ks_mu_loss.item()
                epoch_ks_sigma += ks_sigma_loss.item()
                epoch_norm     += chunk_norm.item()
                epoch_rep  += rep_loss.item()
                n_chunks   += 1
                global_step += 1

                if experiment and global_step % config['log_every'] == 0:
                    if all_gate_scores:
                        sc = torch.cat([s.detach().flatten() for s in all_gate_scores])
                        experiment.log_metric('gate_mean', sc.mean().item(), step=global_step)
                        experiment.log_metric('gate_std',  sc.std().item(),  step=global_step)
                    h_norm = torch.norm(h, dim=-1).mean().item()
                    experiment.log_metric('h_norm_mean', h_norm, step=global_step)
                    experiment.log_metric('loss_lm',    chunk_lm.item(),  step=global_step)
                    experiment.log_metric('loss_ks_mu',    ks_mu_loss.item(),    step=global_step)
                    experiment.log_metric('loss_ks_sigma', ks_sigma_loss.item(), step=global_step)
                    experiment.log_metric('loss_norm',  chunk_norm.item(),step=global_step)
                    experiment.log_metric('loss_rep',   rep_loss.item(),  step=global_step)
                    experiment.log_metric('loss_total', loss.item(),      step=global_step)
                    experiment.log_metric('lr', scheduler.get_last_lr()[0], step=global_step)
                    n_routing = n_tok * config['max_depth']
                    experiment.log_metric('routing_dist_mean',
                                          dist_sum / max(n_routing, 1), step=global_step)
                    experiment.log_metric('routing_weight_entropy',
                                          weight_ent / max(n_routing, 1), step=global_step)
                    experiment.log_metric('routing_null_weight_mean',
                                          null_w_sum / max(n_routing, 1), step=global_step)
                    experiment.log_metric('routing_entropy_mean',
                                          chunk_ent.item(), step=global_step)
                    experiment.log_metric('lambda_ent_current',
                                          lambda_ent, step=global_step)
                    total_dispatches = sum(expert_counts.values())
                    for eid, cnt in expert_counts.items():
                        lbl = model.processing_layer.expert_labels[eid]
                        experiment.log_metric(
                            f'expert_{eid}_{lbl}_util',
                            cnt / max(total_dispatches, 1), step=global_step)
                    with torch.no_grad():
                        _as = model.processing_layer.router.active_slots
                        c = model.processing_layer.router.centroids[
                            torch.tensor(_as, device=dev, dtype=torch.long)]
                        if c.shape[0] >= 2:
                            diff = c.unsqueeze(0) - c.unsqueeze(1)
                            experiment.log_metric('centroid_spread',
                                                  diff.norm(dim=-1).mean().item(), step=global_step)

            avg_lm  = epoch_lm  / max(n_chunks, 1)
            avg_ppl = math.exp(min(avg_lm, 20))
            pbar.set_postfix({
                'lm'   : f'{avg_lm:.3f}',
                'ppl'  : f'{avg_ppl:.1f}',
                'ks_mu': f'{epoch_ks_mu/max(n_chunks,1):.4f}',
                'ks_s' : f'{epoch_ks_sigma/max(n_chunks,1):.4f}',
                'pool' : model.processing_layer.n_experts,
            })

        # ── Validation ────────────────────────────────────────────────────
        model.eval()
        val_lm_sum = 0.0
        val_n      = 0
        with torch.no_grad():
            for input_ids, target_ids in tqdm(val_loader, desc='Validation', leave=False):
                input_ids  = input_ids.to(dev)
                target_ids = target_ids.to(dev)
                h = model.init_state(input_ids.shape[0], dev)
                seq_lm = 0.0
                for t in range(input_ids.shape[1]):
                    al, gs, h, _ = model.forward_token(
                        input_ids[:, t], h, config['max_depth']
                    )
                    seq_lm += compute_lm_loss(al, gs, target_ids[:, t],
                                              config['max_depth']).item()
                val_lm_sum += seq_lm / input_ids.shape[1]
                val_n      += 1

        val_lm  = val_lm_sum / max(val_n, 1)
        val_ppl = math.exp(min(val_lm, 20))
        print(f'\nEpoch {epoch+1} | val_loss={val_lm:.4f} | val_ppl={val_ppl:.2f} '
              f'| pool={model.processing_layer.n_experts}')
        if experiment:
            experiment.log_metric('val_loss_lm', val_lm,  epoch=epoch)
            experiment.log_metric('val_ppl',     val_ppl, epoch=epoch)
            experiment.log_metric('pool_size_epoch', model.processing_layer.n_experts, epoch=epoch)

        ckpt = f'adaptive_ssm_moe_p3_epoch{epoch+1}.pt'
        torch.save({
            'epoch'                     : epoch + 1,
            'model_state_dict'          : model.state_dict(),
            'config'                    : config,
            'expert_labels'             : model.processing_layer.expert_labels,
            'lifecycle_n_births'        : lifecycle_manager.n_births,
            'lifecycle_n_deaths'        : lifecycle_manager.n_deaths,
            'val_loss'                  : val_lm,
        }, ckpt)
        print(f'Checkpoint saved: {ckpt}')
        if experiment:
            log_model(experiment, model, f'AdaptiveSSMMoE-P3-epoch{epoch+1}')

    return model

---
## 10. Hyperparameters and Run

**Phase 3 additions vs Phase 2:**
- `n_experts_init = 4` — pool starts small (was fixed 24)
- `max_experts = 24` — hard cap on pool growth
- `birth_threshold = 0.3` — marginal utility score that triggers a birth event
- `death_threshold = 5` — queries per rolling window below which an expert is pruned
- `grace_period = 200` — steps TournamentExpert trains before type is promoted
- `rolling_window_size = 500` — rolling window for query/loss statistics
- `lifecycle_check_interval = 100` — steps between lifecycle evaluations


In [ ]:
config = {
    # Tokenizer
    'tokenizer'    : 'gpt2',

    # Model dimensions
    'd_embed'      : 128,
    'd_state'      : 256,
    'd_internal'   : 256,

    # MoE expert pool
    'd_expert'          : 96,     # hidden dim per SwiGLU expert (≈same param count as 128 single-path)
    'n_experts_init'    : 12,      # pool starts here, grows to max_experts
    'max_experts'       : 24,     # hard cap (fixed pool size for DDP)
    'top_k'             : 2,      # experts activated per routing step

    # Router
    'router_temperature' : 1.0,
    'faiss_threshold'    : 32,    # use FAISS when n_active > this
    'faiss_rebuild_interval': 200,# steps between FAISS index rebuilds

    # Depth curriculum
    'max_depth'    : 1,

    # Sequence & batching
    'seq_len'      : 256,
    'batch_size'   : 256,
    'chunk_size'   : 64,

    # Optimisation
    'epochs'       : 5,
    'lr'           : 3e-4,
    'weight_decay' : 0.1,
    'max_grad_norm': 1.0,

    # Loss weights
    # KS anchoring: separate mu and sigma terms
    # Higher sigma weight (0.10) reflects the observed std=2 issue
    'lambda_ks_mu'     : 0.01,
    'lambda_ks_sigma'  : 0.05,
    'lambda_norm'      : 0.02,
    'lambda_rep'       : 0.01,
    'repulsion_margin' : 2.0,

    # ── Phase 3: lifecycle hyperparameters ──────────────────────────────
    'death_threshold'         : 4,      # weight-sum < this triggers death
    'grace_period'            : 400,    # steps before tournament decides
    'rolling_window_size'     : 500,    # rolling window for stats (entries)
    'lifecycle_check_interval': 400,    # steps between lifecycle checks
    'split_epsilon'           : 0.3,    # centroid offset for child experts
    'split_cooldown'          : 500,    # min steps between splits

    # Data subsets
    'num_train'    : 100_000,
    'num_val'      : 4_000,

    # Entropy regularization
    'lambda_ent_max'   : 0.01,   # maximum entropy reg coefficient
    'ent_warmup_steps' : 8000,  # steps to ramp lambda_ent from 0 to lambda_ent_max

    # Logging
    'log_every'    : 50,
}

print('Config:')
for k, v in config.items():
    print(f'  {k:30s} = {v}')

model = train(config, experiment=experiment)


---
## 11. Sanity Checks

Run these before a full training run to verify lifecycle mechanics work correctly.


In [ ]:
# ── Sanity check 1: Fixed-size pool add/remove ───────────────────────────
print('=== Sanity Check 1: Dynamic pool management ===')
router_test = MoERouter(d_internal=16, n_experts=2, top_k=2, max_experts=8)
assert router_test.n_experts == 2
router_test.add_centroid(torch.randn(16))
assert router_test.n_experts == 3
router_test.remove_centroid(1)     # deactivate slot 1
assert router_test.n_experts == 2
print(f'  Router expert_active: {router_test.expert_active.tolist()[:8]}  ✓')

layer_test = MoEProcessingLayer(
    d_internal=16, d_expert=8, n_experts_init=3, top_k=2, max_experts=8)
assert layer_test.n_experts == 3
new_eid = layer_test.add_expert(
    MLPExpert(16, 8), torch.randn(16), label='test')
assert layer_test.n_experts == 4
layer_test.remove_expert(new_eid)
assert layer_test.n_experts == 3
print(f'  ProcessingLayer add/remove: pool={layer_test.n_experts}  ✓')

# Verify batched dispatch produces same shape as sequential
z_td = torch.randn(4, 16)
z_out, rinfo = layer_test(z_td)
assert z_out.shape == (4, 16), f'Bad output shape: {z_out.shape}'
assert len(rinfo['expert_ids']) == 3
assert rinfo['z_input'].shape == (4, 16)
print(f'  Forward pass shape check  ✓')

# ── Sanity check 2: Tournament forward pass ───────────────────────────────
print('\n=== Sanity Check 2: TournamentExpert ===')
tour = TournamentExpert(d_internal=16, d_expert=8, grace_period=9)
z_test = torch.randn(4, 16)
for step in range(9):
    delta = tour(z_test)
    assert delta.shape == (4, 16), f'Bad delta shape: {delta.shape}'
    tour.record_step_loss(float(step) * 0.01)
assert tour.is_done()
winner_key, winner_module = tour.promote()
assert winner_key in ('gelu', 'silu', 'poly')
print(f'  Tournament winner: {winner_key}  ✓')
print(f'  Phase losses: { {k: round(sum(v)/max(len(v),1), 4) for k,v in tour._phase_losses.items()} }')

# ── Sanity check 3: Lifecycle split dry run ──────────────────────────────
print('\n=== Sanity Check 3: Lifecycle split dry run ===')
dev_test = torch.device('cpu')

layer3 = MoEProcessingLayer(
    d_internal=16, d_expert=8, n_experts_init=2, top_k=2, max_experts=8)
lm3 = ExpertLifecycleManager(
    grace_period=9,
    death_threshold=0,           # death never triggers (w_sum >= 0)
    rolling_window_size=100,
    d_internal=16, d_expert=8,
    split_epsilon=0.3,
    split_cooldown=0,            # no cooldown for this test
)
for eid in layer3._expert_ids:
    lm3.register_expert(eid, 'live', birth_step=0)

# Inject artificial high-Q stats for first expert
eid0 = layer3._expert_ids[0]
for _ in range(26):              # >= rolling_window_size // 4 = 25
    lm3._wl_values[eid0].append(1.0)
    lm3._w_values[eid0].append(0.1)

events = lm3.check_lifecycle(
    global_step=1, processing_layer=layer3, max_experts=8, dev=dev_test)
assert layer3.n_experts == 3, f'Expected 3 experts, got {layer3.n_experts}'
print(f'  Events: {events}')
print(f'  Pool after split: {layer3.n_experts} experts  ✓')
new_eid = layer3._expert_ids[-1]
assert isinstance(layer3.get_expert(new_eid), TournamentExpert), \
    'Newborn should be TournamentExpert'
print(f'  Newborn is TournamentExpert  ✓')

print('\nAll sanity checks passed ✓')


---
## 12. Sample Generation  *(unchanged from Phase 2)*


In [ ]:
def generate_sample(model, prompt: str, tokenizer, max_new_tokens=150,
                    mu=0.0, temperature=0.9, top_k=50):
    dev    = next(model.parameters()).device
    tokens = tokenizer.encode(prompt, return_tensors='pt')
    output = model.generate(tokens, max_new_tokens=max_new_tokens,
                            mu=mu, temperature=temperature, top_k=top_k)
    return tokenizer.decode(output[0], skip_special_tokens=True)


tokenizer = AutoTokenizer.from_pretrained(config['tokenizer'])
tokenizer.pad_token = tokenizer.eos_token

prompt = 'Once upon a time, a little girl found a golden key in the forest.'

print('=' * 60)
print(f'Prompt: {prompt}')
print(f'Pool size: {model.processing_layer.n_experts} experts')
print('=' * 60)

for mu, label in [(-1.0, 'Fast   (mu=-1)'), (0.0, 'Default (mu=0)'), (1.0, 'Deep   (mu=+1)')]:
    print(f'\n--- {label} ---')
    print(generate_sample(model, prompt, tokenizer, mu=mu))
